In [1]:
# Step 1 — Imports, device, and seed
import os, random, math, numpy as np, pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, BatchNorm, global_mean_pool

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.sparse import issparse
from joblib import dump, load

# Reproducibility
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [2]:
# Utility helpers

def make_undirected(edge_index_np):
    """Ensure edges are bidirectional, remove duplicates and self-loops."""
    rows, cols = edge_index_np
    mask = rows != cols
    rows, cols = rows[mask], cols[mask]
    rows2 = np.concatenate([rows, cols])
    cols2 = np.concatenate([cols, rows])
    edges = np.vstack([rows2, cols2]).T
    edges = np.unique(edges, axis=0)
    return edges.T

def fully_connected_edges(n):
    ii, jj = np.triu_indices(n, k=1)
    e = np.vstack([np.concatenate([ii, jj]), np.concatenate([jj, ii])])  # make undirected
    return e

In [3]:
# Step 3 — Load CSV & target

CSV_PATH = "Concrete_Data.csv" 
TARGET_COL = "Concrete compressive strength(MPa, megapascals) "

df = pd.read_csv(CSV_PATH)
assert TARGET_COL in df.columns, f"TARGET_COL '{TARGET_COL}' not in columns: {df.columns.tolist()}"
X_df = df.drop(columns=[TARGET_COL]).copy()   
y_raw = df[TARGET_COL].to_numpy(dtype=float).reshape(-1, 1)

print("Dataset:", df.shape)
print("Features:", X_df.shape)

Dataset: (1030, 9)
Features: (1030, 8)


In [4]:
# Preprocess 

y_raw = df[TARGET_COL].to_numpy().reshape(-1, 1)
X_df = df.drop(columns=[TARGET_COL])

num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]

def build_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", build_ohe())
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop",
    sparse_threshold=0.3
)

X = preprocess.fit_transform(X_df)
if issparse(X):
    X = X.toarray()

from sklearn.preprocessing import StandardScaler as _SS
y_scaler = _SS().fit(y_raw)
y_scaled = y_scaler.transform(y_raw).ravel()

print("X shape:", X.shape, "| y shape:", y_scaled.shape)
print("Numeric cols:", len(num_cols), "| Categorical cols:", len(cat_cols))


X shape: (1030, 8) | y shape: (1030,)
Numeric cols: 8 | Categorical cols: 0


In [5]:
# Build kNN neighborhoods ONCE

k = 3  
nbrs = NearestNeighbors(n_neighbors=min(k+1, X.shape[0]), metric="euclidean")
nbrs.fit(X)
dist, idx_mat = nbrs.kneighbors(X, n_neighbors=min(k+1, X.shape[0]), return_distance=True)

In [6]:
# Step 6 — Build a list of Data graphs 

graphs = []
targets = []
X_tensor = torch.tensor(X, dtype=torch.float32)

for center in range(X.shape[0]):
    nbr_idx = idx_mat[center] 
    local_X = X_tensor[nbr_idx]
    eidx_np = fully_connected_edges(local_X.shape[0])

    edge_index = torch.tensor(eidx_np, dtype=torch.long)
    y_i = torch.tensor([y_scaled[center]], dtype=torch.float32) 

    graphs.append(Data(x=local_X, edge_index=edge_index))
    targets.append(y_i)

for g, t in zip(graphs, targets):
    g.y = t  

print(f"Built {len(graphs)} graphs; example graph:", graphs[0])

Built 1030 graphs; example graph: Data(x=[4, 8], edge_index=[2, 12], y=[1])


In [7]:
# Train/Val/Test splits

N = len(graphs)
indices = np.arange(N)
train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=42)
val_idx, test_idx  = train_test_split(temp_idx, test_size=0.50, random_state=42)

train_graphs = [graphs[i] for i in train_idx]
val_graphs   = [graphs[i] for i in val_idx]
test_graphs  = [graphs[i] for i in test_idx]

print(f"Split -> Train {len(train_graphs)}, Val {len(val_graphs)}, Test {len(test_graphs)}")

Split -> Train 721, Val 154, Test 155


In [8]:
# GraphSAGE model with global readout

class GraphLevelSAGENet(nn.Module):
    def __init__(self, in_channels, hidden=128, layers=3, dropout=0.25):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        self.convs.append(SAGEConv(in_channels, hidden))
        self.bns.append(BatchNorm(hidden))
        for _ in range(layers-2):
            self.convs.append(SAGEConv(hidden, hidden))
            self.bns.append(BatchNorm(hidden))

        self.dropout = dropout
        self.head = nn.Linear(hidden, 1)

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        # graph readout
        g = global_mean_pool(x, batch)     
        out = self.head(g).view(-1)         
        return out


in_dim = graphs[0].num_features
model = GraphLevelSAGENet(in_channels=in_dim, hidden=128, layers=3, dropout=0.25).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

In [9]:
# DataLoaders

batch_size = 64
train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_graphs,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_graphs,  batch_size=batch_size, shuffle=False)

In [10]:
# Train with early stopping

from pathlib import Path
CKPT_DIR = Path.cwd() / "artifacts"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
best_path = CKPT_DIR / "gnn_graph_best.pt"

best_val = float("inf")
patience, bad = 30, 0
history = []

def run_one_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()
    running = 0.0
    n = 0
    for batch in loader:
        batch = batch.to(device)
        if train:
            optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y.view(-1))
        if train:
            loss.backward()
            optimizer.step()
        running += loss.item() * batch.num_graphs
        n += batch.num_graphs
    return running / max(n, 1)

for epoch in range(1, 1001):
    tr_loss = run_one_epoch(train_loader, train=True)
    val_loss = run_one_epoch(val_loader, train=False)
    te_loss  = run_one_epoch(test_loader, train=False)
    history.append((epoch, tr_loss, val_loss, te_loss))

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        bad = 0
        try:
            tmp = CKPT_DIR / f"tmp_epoch{epoch}.pt"
            torch.save(model.state_dict(), tmp.open("wb"))
            tmp.replace(best_path)
        except Exception as e:
            fb = CKPT_DIR / f"gnn_graph_best_epoch{epoch}.pt"
            torch.save(model.state_dict(), fb.open("wb"))
            print(f"[WARN] Could not write {best_path.name}: {e}. Saved fallback: {fb.name}")
    else:
        bad += 1

    if epoch % 25 == 0 or bad == 0:
        print(f"Epoch {epoch:4d} | train {tr_loss:.4f} | val {val_loss:.4f} ")
    if bad >= patience:
        print(f"Early stop at epoch {epoch}. Best val {best_val:.4f}")
        break

# Load best
if best_path.exists():
    model.load_state_dict(torch.load(best_path, map_location=device))
    print(f"Loaded best checkpoint from: {best_path}")
else:
    print("[WARN] No best checkpoint found; using last-epoch weights.")

Epoch    1 | train 0.6294 | val 0.6424 
Epoch    2 | train 0.3841 | val 0.4854 
Epoch    3 | train 0.3325 | val 0.4437 
Epoch    4 | train 0.3168 | val 0.4058 
Epoch    5 | train 0.2871 | val 0.3727 
Epoch    6 | train 0.2863 | val 0.3713 
Epoch    7 | train 0.2720 | val 0.3538 
Epoch    8 | train 0.2800 | val 0.3536 
Epoch   10 | train 0.2532 | val 0.3431 
Epoch   12 | train 0.2455 | val 0.3255 
Epoch   14 | train 0.2613 | val 0.3242 
Epoch   17 | train 0.2435 | val 0.3238 
Epoch   19 | train 0.2263 | val 0.3228 
Epoch   20 | train 0.2125 | val 0.3161 
Epoch   21 | train 0.2312 | val 0.3117 
Epoch   23 | train 0.2302 | val 0.3026 
Epoch   25 | train 0.2123 | val 0.3127 
Epoch   31 | train 0.2262 | val 0.2909 
Epoch   50 | train 0.2120 | val 0.3045 
Epoch   54 | train 0.1796 | val 0.2878 
Epoch   59 | train 0.1878 | val 0.2848 
Epoch   75 | train 0.1686 | val 0.3124 
Epoch   81 | train 0.1810 | val 0.2799 
Epoch  100 | train 0.1754 | val 0.3073 
Epoch  106 | train 0.1764 | val 0.2721 


In [11]:
# Evaluate in original units

import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

@torch.no_grad()
def evaluate_loader(loader, name):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch).detach().cpu().numpy().reshape(-1, 1)
        yb  = batch.y.detach().cpu().numpy().reshape(-1, 1)
        preds.append(out)
        trues.append(yb)
    preds = np.vstack(preds)
    trues = np.vstack(trues)

    # invert target scaling
    y_true = y_scaler.inverse_transform(trues).ravel()
    y_pred = y_scaler.inverse_transform(preds).ravel()

    rmse = (mean_squared_error(y_true, y_pred) ** 0.5)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{name:5s} | RMSE {rmse:.4f} | MAE {mae:.4f} | R² {r2:.4f} | MAPE {mape:.2f}%")
    return rmse, mae, r2, mape 

evaluate_loader(train_loader, "Train")
evaluate_loader(val_loader,   "Val")
evaluate_loader(test_loader,  "Test")


Train | RMSE 5.8838 | MAE 4.3346 | R² 0.8713 | MAPE 16.37%
Val   | RMSE 8.6899 | MAE 6.4690 | R² 0.7856 | MAPE 25.38%
Test  | RMSE 6.9818 | MAE 5.2942 | R² 0.8138 | MAPE 19.48%


(6.9817654302439305, 5.2942266, 0.8138393759727478, 19.476936757564545)

In [12]:
import sys
import torch
import torch_geometric
import sklearn

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.7.1+cpu
PyTorch Geometric: 2.8.0.post1
scikit-learn: 1.5.1
